# Практика 2 — вариант 24
Соколовский Владислав, Ист-05/26б. Базовое извлечение.

In [1]:
import os
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
load_dotenv()
assert os.getenv("GIGA_KEY"), "Создайте локальный .env с GIGA_KEY"
llm = GigaChat(
    credentials=os.environ["GIGA_KEY"],
    model="GigaChat-2",
    temperature=0.2,
    max_tokens=1000,
    verify_ssl_certs=False
)
print(llm.invoke("Привет!").content)

Привет!


In [2]:
prompt = ChatPromptTemplate.from_messages([("system", "Ты эксперт по заявкам на аренду жилья."), ("human", "Заявка: {text}\nОпредели количество проживающих, включая детей. Верни только целое число.")])
chain = prompt | llm | StrOutputParser()
df = pd.read_csv("rental_24.csv", sep=";")
assert len(df) == 15
results = []
for t in df.text:
    try:
        results.append(chain.invoke({"text": t}))
    except Exception as e:
        results.append("ERROR: " + type(e).__name__)
df["result"] = results
display(df)

,amount,text,result
0,2,Ищем жилье на двоих с 7.08-14.08 . Ближе к цен...,2
1,4,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...,4
2,5,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го...",5
3,4,Ищем жильё с15.06 на 2 недели 2 взрослыми 2 де...,4
4,3,Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июл...,3
5,3,Здравствуйте. Интересует жилье 3 местный номер...,3
6,6,Ищем жилье на начало августа. Двое взрослых и ...,7
7,2,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2
8,2,Ищем жилье в Лазаревском с 12.09. по 20.09. Се...,Не хватает информации для точного определения ...
9,3,ищем жилье на 10 дней с 25 июля по 4-5 августа...,3


In [3]:
df["correct"] = pd.to_numeric(df["result"], errors="coerce").eq(df["amount"])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
if df.result.str.startswith("ERROR:").all():
    print("API недоступен: точность не измерена")
else:
    print(f"Верно: {df.correct.sum()}; ошибок: {(~df.correct).sum()}; точность: {df.correct.mean():.1%}")
display(df.loc[~df.correct])

Верно: 11; ошибок: 4; точность: 73.3%


,amount,text,result,correct
6,6,Ищем жилье на начало августа. Двое взрослых и ...,7,False
8,2,Ищем жилье в Лазаревском с 12.09. по 20.09. Се...,Не хватает информации для точного определения ...,False
10,4,"Здравствуйте, интересует жилье п.Лазаревское 2...",Всего 4 человека.,False
12,1,Здравствуйте. Ищем жильё в период с 26 мая по ...,Один человек плюс две собаки – всего **3** про...,False


In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Optional
class Rental(BaseModel):
    count: int = Field(ge=1)
    count_adults: Optional[int] = Field(default=None, ge=0)
    count_children: Optional[int] = Field(default=None, ge=0)
    start_date: Optional[str] = None
    nights: Optional[int] = Field(default=None, ge=0)
    price_per_day: Optional[int] = Field(default=None, ge=0)
    remarks: Optional[str] = None
parser = PydanticOutputParser(pydantic_object=Rental)
few = ChatPromptTemplate.from_messages([
    ("system", "Извлекай количество людей. Примеры: пара = 2; трое студентов = 3; семья с двумя детьми = 4; один человек и собака = 1. Не считай возраст, животных и комнаты. Верни число."),
    ("human", "{text}")])
improved = few | llm | StrOutputParser()
structured = ChatPromptTemplate.from_messages([
    ("system", "Извлекай данные аренды. Примеры: пара = 2 взрослых, 0 детей; 2 взрослых и ребенок = 3 человека; один человек и собака = 1, состав по возрасту неизвестен. Не угадывай возрастной состав. Дата без года в формате MM-DD, если точный день неизвестен — null. Для диапазона заезда бери раннюю дату, для длительности и цены максимум. При датах заезда и выезда вычисляй разницу в ночах; явный диапазон длительности важнее. Цена только за сутки, если единица неясна — null. Неизвестные поля null."),
    ("human", "Заявка: {text}\nВерни JSON. {format_instructions}")]).partial(format_instructions=parser.get_format_instructions())
json_chain = structured | llm | parser
def safe_invoke(c, t):
    try:
        return c.invoke({"text": t})
    except Exception as e:
        return "ERROR: " + type(e).__name__
df["improved"] = [safe_invoke(improved,t) for t in df.text]
df["improved_correct"] = pd.to_numeric(df.improved, errors="coerce").eq(df.amount)
df.to_csv("rental_improved_results.csv", index=False, encoding="utf-8-sig")
for name in ["result","improved"]:
    if df[name].str.startswith("ERROR:").all():
        print(name, "не измерено")
    else:
        print(name, pd.to_numeric(df[name], errors="coerce").eq(df.amount).mean())


result 0.7333333333333333
improved 0.8


In [5]:
truth = pd.read_csv("rental_ground_truth.csv", sep=";")
assert len(truth) == len(df)
records = []
for t in df.text:
    result = safe_invoke(json_chain,t)
    records.append(result.model_dump() if isinstance(result,Rental) else {"error":result})
pred = pd.DataFrame(records).reindex(columns=["count","count_adults","count_children","start_date","nights","price_per_day","remarks","error"])
out = pd.concat([df[["text","amount"]], truth.add_prefix("expected_"), pred.add_prefix("pred_")],axis=1)
out.to_csv("rental_structured_results.csv", index=False, encoding="utf-8-sig")
fields = ["count_adults","count_children","start_date","nights","price_per_day"]
valid = pred.error.isna()
scores = {}
for f in fields:
    a,b = truth[f],pred[f]
    match = (a.eq(b) | (a.isna() & b.isna())) & valid
    scores[f] = match.mean()
if valid.any():
    print(pd.Series(scores).map(lambda x:f"{x:.1%}"))
    print("Средняя точность:", f"{sum(scores.values())/len(scores):.1%}")
else:
    print("Нет валидных JSON-ответов: точность не измерена")
display(out)

count_adults      13.3%
count_children    13.3%
start_date         0.0%
nights            13.3%
price_per_day     13.3%
dtype: str
Средняя точность: 10.7%


,text,amount,expected_count_adults,expected_count_children,expected_start_date,expected_nights,expected_price_per_day,expected_remarks,pred_count,pred_count_adults,pred_count_children,pred_start_date,pred_nights,pred_price_per_day,pred_remarks,pred_error
0,Ищем жилье на двоих с 7.08-14.08 . Ближе к цен...,2,NaN,NaN,08-07,7.0,NaN,"центр, море, Wi-Fi, кондиционер",NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
1,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...,4,3.0,1.0,06-30,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
2,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го...",5,2.0,3.0,07-11,12.0,NaN,"две двуспальные кровати, центр, море, эконом",NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
3,Ищем жильё с15.06 на 2 недели 2 взрослыми 2 де...,4,2.0,2.0,06-15,14.0,NaN,"море, недорого, квартира",NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
4,Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июл...,3,2.0,1.0,07-08,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
5,Здравствуйте. Интересует жилье 3 местный номер...,3,NaN,NaN,06-20,8.0,NaN,"трехместный номер, море",NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
6,Ищем жилье на начало августа. Двое взрослых и ...,6,2.0,4.0,NaN,NaN,NaN,отдельная квартира,6.0,2.0,4.0,08-01,NaN,NaN,NaN,NaN
7,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,NaN,NaN,08-07,8.0,NaN,"эконом, море, писать в личку",NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException
8,Ищем жилье в Лазаревском с 12.09. по 20.09. Се...,2,2.0,0.0,09-12,8.0,NaN,частный сектор,2.0,2.0,0.0,12-09,8.0,NaN,Only private sector,NaN
9,ищем жилье на 10 дней с 25 июля по 4-5 августа...,3,NaN,NaN,07-25,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR: OutputParserException


In [6]:
import json
from langchain_core.output_parsers import StrOutputParser
structured_v2 = ChatPromptTemplate.from_messages([
    ("system", """Ты извлекаешь данные заявок на жильё. Ответ строго один JSON-объект без Markdown и пояснений.
Ключи: count, count_adults, count_children, start_date, nights, price_per_day, remarks.
Числа — целые, неизвестные значения — null. Дата MM-DD (месяц, затем день), например 12 сентября = 09-12. Если день неизвестен, null.
Учитывай вместимость номера, не считай животных. Для диапазонов дат бери ранний заезд и максимальную длительность; цену бери максимальную за сутки. Не угадывай возрастной состав.
Пример: Два взрослых и ребёнок с 12 сентября на 5 ночей, до 3000 руб/сут, нужен бассейн.
Ответ: {{"count":3,"count_adults":2,"count_children":1,"start_date":"09-12","nights":5,"price_per_day":3000,"remarks":"бассейн"}}"""),
    ("human", "{text}")
])
raw_chain = structured_v2 | llm | StrOutputParser()
def extract_v2(t):
    raw = ""
    for attempt in range(2):
        try:
            raw = raw_chain.invoke({"text":t})
            start, end = raw.index("{"), raw.rindex("}")+1
            obj = json.loads(raw[start:end])
            return Rental.model_validate(obj).model_dump() | {"raw_response":raw}
        except Exception as e:
            error = type(e).__name__
    return {"error":error,"raw_response":raw}

truth = pd.read_csv("rental_ground_truth.csv", sep=";")
assert len(truth) == len(df)
records = []
for t in df.text:
    records.append(extract_v2(t))
pred = pd.DataFrame(records).reindex(columns=["count","count_adults","count_children","start_date","nights","price_per_day","remarks","error","raw_response"])
out = pd.concat([df[["text","amount"]], truth.add_prefix("expected_"), pred.add_prefix("pred_")],axis=1)
out.to_csv("rental_structured_results_v2.csv", index=False, encoding="utf-8-sig")
fields = ["count_adults","count_children","start_date","nights","price_per_day"]
valid = pred.error.isna()
scores = {}
for f in fields:
    a,b = truth[f],pred[f]
    match = (a.eq(b) | (a.isna() & b.isna())) & valid
    scores[f] = match.mean()
if valid.any():
    print(pd.Series(scores).map(lambda x:f"{x:.1%}"))
    print("Средняя точность:", f"{sum(scores.values())/len(scores):.1%}")
else:
    print("Нет валидных JSON-ответов: точность не измерена")
print("Валидных JSON:", int(valid.sum()), "из", len(df))
display(out)

count_adults      60.0%
count_children    86.7%
start_date        73.3%
nights            60.0%
price_per_day     80.0%
dtype: str
Средняя точность: 72.0%
Валидных JSON: 13 из 15


,text,amount,expected_count_adults,expected_count_children,expected_start_date,expected_nights,expected_price_per_day,expected_remarks,pred_count,pred_count_adults,pred_count_children,pred_start_date,pred_nights,pred_price_per_day,pred_remarks,pred_error,pred_raw_response
0,Ищем жилье на двоих с 7.08-14.08 . Ближе к цен...,2,NaN,NaN,08-07,7.0,NaN,"центр, море, Wi-Fi, кондиционер",2.0,2.0,NaN,08-07,7.0,NaN,"центр, море, Wi-Fi, кондиционер",NaN,"{""count"":2,""count_adults"":2,""start_date"":""08-0..."
1,Снимем жильё с 30 июня по 7 июля 3 взрослых и ...,4,3.0,1.0,06-30,7.0,NaN,NaN,4.0,3.0,1.0,06-30,7.0,NaN,NaN,NaN,"{""count"":4,""count_adults"":3,""count_children"":1..."
2,"Ищем жилье! 2взрослых и 3 ребенка ( 6,6 и 4 го...",5,2.0,3.0,07-11,12.0,NaN,"две двуспальные кровати, центр, море, эконом",5.0,2.0,3.0,07-11,12.0,NaN,NaN,NaN,"{""count"":5,""count_adults"":2,""count_children"":3..."
3,Ищем жильё с15.06 на 2 недели 2 взрослыми 2 де...,4,2.0,2.0,06-15,14.0,NaN,"море, недорого, квартира",4.0,2.0,2.0,06-15,14.0,NaN,NaN,NaN,"{""count"":4,""count_adults"":2,""count_children"":2..."
4,Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июл...,3,2.0,1.0,07-08,13.0,NaN,NaN,3.0,2.0,1.0,07-08,14.0,NaN,NaN,NaN,"{""count"":3,""count_adults"":2,""count_children"":1..."
5,Здравствуйте. Интересует жилье 3 местный номер...,3,NaN,NaN,06-20,8.0,NaN,"трехместный номер, море",NaN,NaN,NaN,NaN,NaN,NaN,NaN,JSONDecodeError,"{""count"":3,""start_date"":""06-20"",""nights"":8,nul..."
6,Ищем жилье на начало августа. Двое взрослых и ...,6,2.0,4.0,NaN,NaN,NaN,отдельная квартира,7.0,2.0,4.0,08-01,NaN,NaN,NaN,NaN,"{""count"":7,""count_adults"":2,""count_children"":4..."
7,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,NaN,NaN,08-07,8.0,NaN,"эконом, море, писать в личку",2.0,2.0,NaN,08-07,8.0,1000.0,NaN,NaN,"{""count"":2,""count_adults"":2,""start_date"":""08-0..."
8,Ищем жилье в Лазаревском с 12.09. по 20.09. Се...,2,2.0,0.0,09-12,8.0,NaN,частный сектор,2.0,2.0,0.0,09-12,8.0,NaN,NaN,NaN,"{""count"":2,""count_adults"":2,""count_children"":0..."
9,ищем жилье на 10 дней с 25 июля по 4-5 августа...,3,NaN,NaN,07-25,10.0,NaN,NaN,3.0,3.0,NaN,07-25,10.0,NaN,NaN,NaN,"{""count"":3,""count_adults"":3,""start_date"":""07-2..."
